In [37]:
import os
import sys
import datasets
sys.path.insert(0, "/home/aiscuser/verl")
from verl.utils.hdfs_io import copy, makedirs
import argparse

import re

from verl.utils.hdfs_io import copy, makedirs
import argparse

from verl.utils.reward_score.math import remove_boxed, last_boxed_only_string
from verl.workers.reward_manager import DAPORewardManager
from transformers import AutoTokenizer, AutoProcessor
from verl.utils.dataset.rl_dataset import RLHFDataset, collate_fn

In [39]:
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-3.1-8B-Instruct')
processor = AutoProcessor.from_pretrained('meta-llama/Llama-3.1-8B-Instruct')
dataset = RLHFDataset(
    parquet_files=['/home/aiscuser/data/dapo-math-17k.parquet'],
    tokenizer=tokenizer,
    processor=processor,
    prompt_key='prompt',
    max_prompt_length=2048,
    filter_prompts=True,
    return_raw_chat=False,
    truncation='error',
    filter_overlong_prompts=False
    )

dataset len: 1791700


In [65]:
import pyarrow.parquet as pq

data = pq.read_table('/home/aiscuser/data/dapo-math-17k.parquet')
data['reward_model']

[
  -- is_valid: all not null
  -- child 0 type: string
    [
      "34",
      "113",
      "-3",
      "3",
      "37",
      ...
      "61",
      "750",
      "400",
      "182",
      "480"
    ]
  -- child 1 type: string
    [
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      ...
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2"
    ],
  -- is_valid: all not null
  -- child 0 type: string
    [
      "588",
      "70",
      "112",
      "486",
      "19",
      ...
      "39",
      "8",
      "76",
      "655",
      "41"
    ]
  -- child 1 type: string
    [
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      "rule-lighteval/MATH_v2",
      ...
      "rule-lighteval/MATH_v2",

In [ ]:
def extract_solution(solution_str):
    solution = re.search("#### (\\-?[0-9\\.\\,]+)", solution_str)
    assert solution is not None
    final_solution = solution.group(0)
    final_solution = final_solution.split('#### ')[1].replace(',', '')
    return final_solution
    
def prepare_gsm8k():
    data_source = 'openai/gsm8k'

    dataset = datasets.load_dataset(data_source, 'main')

    dataset = dataset['test']

    instruction_following_1 = 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n'
    instruction_following_2 = 'Remember to put your answer on its own line after "Answer:".'
    # add a row to each data item that represents a unique id
    def make_map_fn(split):

        def process_fn(example, idx):
            question_raw = example.pop('question')

            question = instruction_following_1 + '\n' + question_raw + '\n' + instruction_following_2

            answer_raw = example.pop('answer')
            solution = extract_solution(answer_raw)
            data = {
                "data_source": 'dapo-math',
                "prompt": [{
                    "role": "user",
                    "content": question,
                }],
                "ability": "MATH",
                "reward_model": {
                    "style": "rule-lighteval/MATH_v2",
                    "ground_truth": solution
                },
                "extra_info": {
                    'dummy': 'dummy',
                }
            }
            return data

        return process_fn

    dataset = dataset.map(function=make_map_fn('test'), with_indices=True)
    print(f'dataset size: {len(dataset)}')
    return dataset

gsm8k_dataset = prepare_gsm8k()
gsm8k_dataset.to_parquet('/home/aiscuser/data/gsm8k_eval.parquet')

Map: 100%|██████████| 1319/1319 [00:00<00:00, 24090.19 examples/s]


dataset size: 1319


Creating parquet from Arrow format: 100%|██████████| 2/2 [00:00<00:00, 619.95ba/s]


735049

In [108]:
def prepare_math500():
    data_source = 'HuggingFaceH4/MATH-500'

    dataset = datasets.load_dataset(data_source, 'default')

    dataset = dataset['test']

    instruction_following_1 = 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n'
    instruction_following_2 = 'Remember to put your answer on its own line after "Answer:".'
    # add a row to each data item that represents a unique id
    def make_map_fn(split):

        def process_fn(example, idx):
            question_raw = example.pop('problem')

            question = instruction_following_1 + '\n' + question_raw + '\n' + instruction_following_2

            answer_raw = example.pop('answer')
            data = {
                "data_source": 'dapo-math',
                "prompt": [{
                    "role": "user",
                    "content": question,
                }],
                "ability": "MATH",
                "reward_model": {
                    "style": "rule-lighteval/MATH_v2",
                    "ground_truth": answer_raw
                },
                "extra_info": {
                    'dummy': 'dummy',
                }
            }
            return data

        return process_fn

    dataset = dataset.map(function=make_map_fn('test'), with_indices=True)
    return dataset

math500_dataset = prepare_math500()
math500_dataset.to_parquet('/home/aiscuser/data/math500_eval.parquet')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 380.95ba/s]


553774

In [109]:
def prepare_minerva():
    data_source = 'zwhe99/simplerl-minerva-math'

    dataset = datasets.load_dataset(data_source, 'default')

    dataset = dataset['test']

    instruction_following_1 = 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n'
    instruction_following_2 = 'Remember to put your answer on its own line after "Answer:".'
    # add a row to each data item that represents a unique id
    def make_map_fn(split):

        def process_fn(example, idx):
            question_raw = example.pop('problem')

            question = instruction_following_1 + '\n' + question_raw + '\n' + instruction_following_2

            answer_raw = example.pop('answer')
            data = {
                "data_source": 'dapo-math',
                "prompt": [{
                    "role": "user",
                    "content": question,
                }],
                "ability": "MATH",
                "reward_model": {
                    "style": "rule-lighteval/MATH_v2",
                    "ground_truth": answer_raw
                },
                "extra_info": {
                    'dummy': 'dummy',
                }
            }
            return data

        return process_fn

    dataset = dataset.map(function=make_map_fn('test'), with_indices=True)
    return dataset
minerva_dataset = prepare_minerva()
minerva_dataset.to_parquet('/home/aiscuser/data/minerva_eval.parquet')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 588.18ba/s]


315378

In [110]:
def prepare_olympiad():
    data_source = 'knoveleng/OlympiadBench'

    dataset = datasets.load_dataset('Hothan/OlympiadBench', 'OE_TO_maths_en_COMP')

    dataset = dataset['train']

    instruction_following_1 = 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n'
    instruction_following_2 = 'Remember to put your answer on its own line after "Answer:".'
    # add a row to each data item that represents a unique id
    def make_map_fn(split):

        def process_fn(example, idx):
            question_raw = example.pop('question')

            question = instruction_following_1 + '\n' + question_raw + '\n' + instruction_following_2

            answer_raw = example.pop('final_answer')[0]
            data = {
                "data_source": 'dapo-math',
                "prompt": [{
                    "role": "user",
                    "content": question,
                }],
                "ability": "MATH",
                "reward_model": {
                    "style": "rule-lighteval/MATH_v2",
                    "ground_truth": answer_raw
                },
                "extra_info": {
                    'dummy': 'dummy',
                }
            }
            return data

        return process_fn

    dataset = dataset.map(function=make_map_fn('test'), with_indices=True)
    dataset = dataset.remove_columns(['solution'])
    # dataset.cast_column("prompt", datasets.Sequence(datasets.Value(dty
    return dataset
olympiad_dataset = prepare_olympiad()
olympiad_dataset.to_parquet('/home/aiscuser/data/olympiad_eval.parquet')

Creating parquet from Arrow format: 100%|██████████| 7/7 [00:00<00:00, 1042.77ba/s]


529992

In [97]:
olympiad_dataset.features

{'id': Value(dtype='int64', id=None),
 'context': Value(dtype='string', id=None),
 'image_1': Image(mode=None, decode=True, id=None),
 'image_2': Image(mode=None, decode=True, id=None),
 'image_3': Image(mode=None, decode=True, id=None),
 'image_4': Image(mode=None, decode=True, id=None),
 'image_5': Image(mode=None, decode=True, id=None),
 'modality': Value(dtype='string', id=None),
 'difficulty': Value(dtype='string', id=None),
 'is_multiple_answer': Value(dtype='bool', id=None),
 'unit': Value(dtype='string', id=None),
 'answer_type': Value(dtype='string', id=None),
 'error': Value(dtype='string', id=None),
 'question_type': Value(dtype='string', id=None),
 'subfield': Value(dtype='string', id=None),
 'subject': Value(dtype='string', id=None),
 'language': Value(dtype='string', id=None),
 'data_source': Value(dtype='string', id=None),
 'prompt': [{'content': Value(dtype='string', id=None),
   'role': Value(dtype='string', id=None)}],
 'ability': Value(dtype='string', id=None),
 'rew

In [ ]:
# merge all datasets
all_datasets = datasets.concatenate_datasets([gsm8k_dataset, math500_dataset, minerva_dataset, olympiad_dataset])
